# 03 — Simple Trend-Following Signal and Backtest

## Why this project exists

A trader may ask: **"Does a simple trend filter improve risk-adjusted returns compared with buy-and-hold?"**

This notebook tests that question using a transparent 50/200 day moving average signal on SPY. The goal is not to claim a profitable strategy. The goal is to show disciplined hypothesis testing, clean implementation and honest interpretation.

In [ ]:
!pip install yfinance plotly -q

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.express as px

pd.options.display.float_format = "{:,.4f}".format

## 1. Download data

I use adjusted prices to avoid distortions from dividends and splits. The backtest is deliberately simple and should be treated as a prototype.

In [ ]:
ticker = "SPY"
prices = yf.download(ticker, start="2005-01-01", auto_adjust=True, progress=False)["Close"]
prices = prices.dropna()
prices.tail()

## 2. Build the signal

The hypothesis:

- If price is above both the 50-day and 200-day moving averages, stay invested.
- Otherwise, move to cash.

To avoid look-ahead bias, I shift the signal by one day before applying it to returns.

In [ ]:
df = pd.DataFrame(index=prices.index)
df["price"] = prices
df["return"] = df["price"].pct_change()
df["ma_50"] = df["price"].rolling(50).mean()
df["ma_200"] = df["price"].rolling(200).mean()
df["signal_raw"] = ((df["price"] > df["ma_50"]) & (df["price"] > df["ma_200"])).astype(int)
df["signal"] = df["signal_raw"].shift(1)
df["strategy_return"] = df["signal"] * df["return"]
df = df.dropna()
df.tail()

## 3. Evaluate performance

I compare the signal against buy-and-hold using:

- Annualized return
- Annualized volatility
- Sharpe ratio
- Max drawdown

I keep the metrics simple because the first objective is to check whether the idea deserves deeper research.

In [ ]:
def max_drawdown(equity):
    return (equity / equity.cummax() - 1).min()

def summarize(ret):
    equity = (1 + ret).cumprod()
    ann_return = ret.mean() * 252
    ann_vol = ret.std() * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan
    return pd.Series({
        "annual_return": ann_return,
        "annual_volatility": ann_vol,
        "sharpe_ratio": sharpe,
        "max_drawdown": max_drawdown(equity),
        "days": len(ret)
    })

summary = pd.DataFrame({
    "buy_and_hold": summarize(df["return"]),
    "trend_signal": summarize(df["strategy_return"])
}).T
summary

## 4. Plot equity curves

Visual inspection is important. A strategy can have a decent average return but still be psychologically difficult to hold.

In [ ]:
equity = pd.DataFrame({
    "buy_and_hold": (1 + df["return"]).cumprod(),
    "trend_signal": (1 + df["strategy_return"]).cumprod()
})
px.line(equity, title="Equity curve: buy-and-hold vs trend signal").show()

signals = df[["price", "ma_50", "ma_200"]].copy()
px.line(signals, title="SPY price with moving averages").show()

## 5. My conclusion

A good quant support analyst should not oversell a simple backtest.

What this notebook demonstrates:

- I can implement a hypothesis without look-ahead bias.
- I can compare a strategy against a benchmark.
- I can calculate risk and return metrics.
- I can communicate limitations clearly.

Limitations:

- No transaction costs
- No slippage
- No tax impact
- No parameter robustness testing
- Only one asset tested

## Possible next iterations

- Test multiple ETFs
- Add transaction costs
- Add walk-forward validation
- Compare against volatility targeting
- Build an interactive Streamlit dashboard